In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from src.spark_session import get_spark
from src.config import PRODUCTS_RAW_PATH, PRODUCTS_SILVER_PATH

In [2]:
spark = get_spark("SilverProducts")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 13:05:46 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/06 13:05:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 13:05:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [3]:
products_raw_df = spark.read.option("header", True).option("inferSchema", False).csv(str(PRODUCTS_RAW_PATH))
products_raw_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [4]:
products_raw_df.show(5, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |1000            |30               |18               |20              |
|96bd76ec8810374ed1b65e291975717f|e

In [6]:
print("Rows: ", products_raw_df.count())
print("Columns: ", len(products_raw_df.columns))
print("Column names: ", products_raw_df.columns)

Rows:  32951
Columns:  9
Column names:  ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [7]:
unique_products_count = products_raw_df.select("product_id").distinct().count()
print("Unique product count: ", unique_products_count)
print("Each row has a unique product_id: ", unique_products_count == products_raw_df.count())

Unique product count:  32951
Each row has a unique product_id:  True


In [8]:
products_raw_df.select(
    F.sum(
        F.col("product_category_name")
        .isNull()
        .cast("int")
    ).alias("null_category_count")
).show()

+-------------------+
|null_category_count|
+-------------------+
|                610|
+-------------------+



In [9]:
products_schema = StructType([
    StructField("product_id", StringType(), nullable=False),
    StructField("product_category_name", StringType(), nullable=True),
    StructField("product_name_lenght", IntegerType(), nullable=True),
    StructField("product_description_lenght", IntegerType(), nullable=True),
    StructField("product_photos_qty", IntegerType(), nullable=True),
    StructField("product_weight_g", IntegerType(), nullable=True),
    StructField("product_length_cm", IntegerType(), nullable=True),
    StructField("product_height_cm", IntegerType(), nullable=True),
    StructField("product_width_cm", IntegerType(), nullable=True),
])

In [12]:
products_typed_df = spark.read.option("header", True).schema(products_schema).csv(str(PRODUCTS_RAW_PATH))
products_typed_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [13]:
products_clean_df = (
    products_typed_df
    .filter(F.col("product_id").isNotNull())
    .withColumn(
        "product_category_name",
        F.coalesce(F.lower(F.trim(F.col("product_category_name"))), F.lit("unknown")),
    )
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .dropDuplicates(["product_id"])
)

products_clean_df.show(5, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|00066f42aeeb9f3007548bb9d3f33c38|perfumaria           |53                 |596                       |6                 |300             |20               |16               |16              |
|00088930e925c41fd95ebfe695fd2655|automotivo           |56                 |752                       |4                 |1225            |55               |10               |26              |
|0011c512eb256aa0dbbb544d8dffcf6e|a

In [14]:
products_clean_df.select(
    "product_id",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
     "product_weight_g",
     "product_length_cm",
     "product_height_cm",
     "product_width_cm"
).filter(F.col("product_category_name") == "unknown").show(10, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0103863bf3441460142ec23c74388e4c|unknown              |NULL               |NULL                      |NULL              |200             |16               |2                |11              |
|014fcf6bd5cd4c7ee29fb3bb618c445e|unknown              |NULL               |NULL                      |NULL              |7000            |55               |30               |45              |
|0292b46c30348496b1be04eeb440bdb5|u

In [16]:
products_clean_df.coalesce(2).write.mode("overwrite").parquet(str(PRODUCTS_SILVER_PATH))

In [17]:
products_by_category = (
    products_typed_df
    .groupBy("product_category_name")
    .count()
    .orderBy(F.col("count").desc())
)

products_by_category.show(20, truncate=False)

+---------------------------------+-----+
|product_category_name            |count|
+---------------------------------+-----+
|cama_mesa_banho                  |3029 |
|esporte_lazer                    |2867 |
|moveis_decoracao                 |2657 |
|beleza_saude                     |2444 |
|utilidades_domesticas            |2335 |
|automotivo                       |1900 |
|informatica_acessorios           |1639 |
|brinquedos                       |1411 |
|relogios_presentes               |1329 |
|telefonia                        |1134 |
|bebes                            |919  |
|perfumaria                       |868  |
|fashion_bolsas_e_acessorios      |849  |
|papelaria                        |849  |
|cool_stuff                       |789  |
|ferramentas_jardim               |753  |
|pet_shop                         |719  |
|NULL                             |610  |
|eletronicos                      |517  |
|construcao_ferramentas_construcao|400  |
+---------------------------------